# 7e - Normalize Extracted Frames (Python/OpenCV)

Skrypt normalizuje klatki wycięte wcześniej przez skrypt 7d.
Odczytuje pliki `.jpg`, zmienia ich rozmiar do `224x224` i konwertuje na odcienie szarości.

In [1]:
import cv2
import pandas as pd
import numpy as np
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import time


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
INPUT_CSV = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_topn_frames_raw.csv'
OUTPUT_FRAMES_DIR = PROJECT_ROOT / 'normalized_frames' / 'topn_224_gray_jpg_from_raw'
OUTPUT_METADATA_CSV = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_topn_frames_normalized_from_raw.csv'

TARGET_SIZE = (224, 224)
MAX_WORKERS = 12


In [3]:
def process_video_frames(row):
    in_path = PROJECT_ROOT / row.get('frames_path', '')
    out_path = OUTPUT_FRAMES_DIR / in_path.parts[-2] / in_path.parts[-1]
    
    if not in_path.exists():
        return row
        
    out_path.mkdir(parents=True, exist_ok=True)
    frames = list(in_path.glob('*.jpg'))
    
    for f in frames:
        img = cv2.imread(str(f))
        if img is not None:
            img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            img_resized = cv2.resize(img_gray, TARGET_SIZE)
            cv2.imwrite(str(out_path / f.name), img_resized)
            
    row['frames_path'] = str(out_path.relative_to(PROJECT_ROOT)).replace('\\', '/')
    row['normalized_width'] = TARGET_SIZE[0]
    row['normalized_height'] = TARGET_SIZE[1]
    row['normalized_color_mode'] = 'grayscale'
    return row


In [4]:
if not INPUT_CSV.exists():
    print(f'Brak pliku wejściowego: {INPUT_CSV}. Uruchom skrypt 7d najpierw.')
else:
    df = pd.read_csv(INPUT_CSV)
    print(f'Rekordów: {len(df)}')

    records = []
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_video_frames, row.to_dict()) for _, row in df.iterrows()]
        for i, fut in enumerate(as_completed(futures)):
            records.append(fut.result())
            if (i+1) % 100 == 0:
                print(f'Przetworzono {i+1}/{len(futures)}')
                
    t1 = time.time()
    print(f'Zakończono w {t1-t0:.1f} sekund.')
    result_df = pd.DataFrame(records)
    result_df.to_csv(OUTPUT_METADATA_CSV, index=False)
    print(f'Zapisano metadane: {OUTPUT_METADATA_CSV}')


Rekordów: 1647
Przetworzono 100/1647
Przetworzono 200/1647
Przetworzono 300/1647
Przetworzono 400/1647
Przetworzono 500/1647
Przetworzono 600/1647
Przetworzono 700/1647
Przetworzono 800/1647
Przetworzono 900/1647
Przetworzono 1000/1647
Przetworzono 1100/1647
Przetworzono 1200/1647
Przetworzono 1300/1647
Przetworzono 1400/1647
Przetworzono 1500/1647
Przetworzono 1600/1647
Zakończono w 35.2 sekund.
Zapisano metadane: C:\Users\kacpe\source\repos\szum\merged_datasets\universal_metadata_topn_frames_normalized_from_raw.csv
